In [8]:
import os
from pyspark.sql import SparkSession, functions as F

# Имя каталога
catalog = "lk"

# Доступ к minio
access_key = os.getenv("MINIO_ROOT_USER", "minioadmin")
secret_key = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
warehouse = os.getenv("LAKEKEEPER_WAREHOUSE", "mydatalab")

#Настройка каталога в Spark
spark = (
    SparkSession.builder.appName("lakekeeper-iceberg-demo")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{catalog}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{catalog}.type", "rest")
    .config(f"spark.sql.catalog.{catalog}.uri", "http://127.0.0.1:8181/catalog")
    .config(f"spark.sql.catalog.{catalog}.warehouse", warehouse)
    .config(f"spark.sql.catalog.{catalog}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{catalog}.s3.endpoint", "http://127.0.0.1:9000")
    .config(f"spark.sql.catalog.{catalog}.s3.path-style-access", "true")
    .config(f"spark.sql.catalog.{catalog}.s3.access-key-id", access_key)
    .config(f"spark.sql.catalog.{catalog}.s3.secret-access-key", secret_key)
    .config(f"spark.sql.legacy.parquet.nanosAsLong","true") #Добавили для
    .config("spark.sql.defaultCatalog", catalog)
    .getOrCreate()
)

#Уровень логирования
spark.sparkContext.setLogLevel("WARN")

In [2]:
# Создание namespace stage
spark.sql("CREATE NAMESPACE IF NOT EXISTS lk.stage")

DataFrame[]

In [ ]:
from pyspark import SparkFiles #Для локальной файловой системы

In [4]:
# Загрузка users
## Скачиваем файл для обработки
spark.sparkContext.addFile("https://inzhenerka-public.s3.eu-west-1.amazonaws.com/scooters_data_generator/users.parquet")
## Загружаем и обрабатываем файл
users_df = spark.read.parquet('file://' + SparkFiles.get('users.parquet'))
## сохраняем фрейм данных в таблицу
users_df.writeTo("lk.stage.users").create()

In [6]:
# Загрузка payments
## Скачиваем файл для обработки
spark.sparkContext.addFile("https://inzhenerka-public.s3.eu-west-1.amazonaws.com/scooters_data_generator/payments.parquet")
## Загружаем и обрабатываем файл
payments_df = spark.read.parquet('file://' + SparkFiles.get('payments.parquet'))
## сохраняем фрейм данных в таблицу
payments_df.writeTo("lk.stage.payments").create()

In [10]:
# Загрузка trips
## Скачиваем файл для обработки
spark.sparkContext.addFile("https://inzhenerka-public.s3.eu-west-1.amazonaws.com/scooters_data_generator/trips.parquet")
## Загружаем и обрабатываем файл
trips_df = spark.read.parquet('file://' + SparkFiles.get('trips.parquet'))
## сохраняем фрейм данных в таблицу
trips_df.writeTo("lk.stage.trips").create()

In [11]:
# Загрузка events
## Скачиваем файл для обработки
spark.sparkContext.addFile("https://inzhenerka-public.s3.eu-west-1.amazonaws.com/scooters_data_generator/events.parquet")
## Загружаем и обрабатываем файл
events_df = spark.read.parquet('file://' + SparkFiles.get('events.parquet'))
## сохраняем фрейм данных в таблицу
events_df.writeTo("lk.stage.events").create()